# Decode data

`data/processed/importaciones_2019-2026_reshaped.parquet` stores one row per declaration item, with every column still a raw string. Several columns are AFIP codes that need to be resolved against the catalogs in `docs/`.

Column -> catalog mapping:

| Column | Catalog | Notes |
|---|---|---|
| `ADU` | `ANEXOIV_Aduanas.csv` | Aduana de registro |
| `M` | `ANEXOV_Vias_y_Medios_de_Transporte.csv` | Medio de transporte |
| `UN` | `ANEXOXV_Unidades_de_medida.csv` | Unidad de medida (uses `Descripcion Ampliada`, since `Descripcion` is inconsistent/abbreviated) |
| `DIV` | `ANEXOXI_Divisas.csv` | Divisa |
| `PAI` (1st) | `ANEXOVII_Paises.csv` | Renamed to `PAIS_ORIGEN` |
| `PAI_duplicated_0` (2nd) | `ANEXOVII_Paises.csv` | Renamed to `PAIS_PROCEDENCIA` |

`COD` (tax concept) is not decoded here: 0.3 already summed `MONTO` across tax concepts and dropped `COD`, so there's nothing left to resolve against `ANEXOXX_Conceptos_Recaudatorios.csv`.

Numeric columns (`CANT_UNIDAD_MEDIDA`, `FOB_DOLAR`, `FOB_TOTAL`) are cast from padded strings to `Float64`; empty strings become null. `MONTO_TRIBUTADO_TOTAL` was already cast and summed in 0.3.

0.3's reshape shrinks the dataset from hundreds of millions of rows down to roughly 30M declarations before it gets here, so the catalog joins below run in a single pass -- no per-year chunking needed.


In [1]:
import logging
from pathlib import Path

import polars as pl

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# Paths
docs_dir = Path("../docs")
processed_dir = Path("../data/processed")
source_path = processed_dir / "importaciones_2019-2026_reshaped.parquet"
output_path = processed_dir / "importaciones_2019-2026_decoded.parquet"


# Load catalogs
# `Codigo` must be forced to Utf8 -- otherwise polars infers it as an integer and strips
# the leading zeros that the raw codes rely on (e.g. "001" -> 1), breaking the join.
def load_catalog(filename: str, desc_col: str = "Descripcion") -> pl.DataFrame:
    return pl.read_csv(
        docs_dir / filename,
        separator="|",
        quote_char='"',
        schema_overrides={"Codigo": pl.Utf8},
    ).select("Codigo", desc_col)


aduanas = load_catalog("ANEXOIV_Aduanas.csv")
transporte = load_catalog("ANEXOV_Vias_y_Medios_de_Transporte.csv")
unidades = load_catalog(
    "ANEXOXV_Unidades_de_medida.csv", desc_col="Descripcion Ampliada"
)
divisas = load_catalog("ANEXOXI_Divisas.csv", desc_col="Denominacion")
paises = load_catalog("ANEXOVII_Paises.csv", desc_col="Denominacion")


# Decode helper
# Left-joins a code column against a catalog and replaces the code in place with its
# description. Codes with no catalog match (e.g. blank M when no transport medium was
# recorded) become null rather than failing the join.
def decode_column(
    lf: pl.LazyFrame, column: str, catalog: pl.DataFrame, desc_col: str = "Descripcion"
) -> pl.LazyFrame:
    lookup = catalog.rename({"Codigo": column, desc_col: f"__{column}_decoded"})
    lf = lf.join(lookup.lazy(), on=column, how="left")
    return lf.with_columns(pl.col(f"__{column}_decoded").alias(column)).drop(
        f"__{column}_decoded"
    )


lf = pl.scan_parquet(source_path)

# Clarify the two PAI columns before decoding, matching declaration order
# (pais de origen listed before pais de procedencia).
lf = lf.rename({"PAI": "PAIS_ORIGEN", "PAI_duplicated_0": "PAIS_PROCEDENCIA"})

lf = decode_column(lf, "ADU", aduanas)
lf = decode_column(lf, "M", transporte)
lf = decode_column(lf, "UN", unidades, desc_col="Descripcion Ampliada")
lf = decode_column(lf, "DIV", divisas, desc_col="Denominacion")
lf = decode_column(lf, "PAIS_ORIGEN", paises, desc_col="Denominacion")
lf = decode_column(lf, "PAIS_PROCEDENCIA", paises, desc_col="Denominacion")

numeric_columns = ["CANT_UNIDAD_MEDIDA", "FOB_DOLAR", "FOB_TOTAL"]
lf = lf.with_columns(
    [pl.col(c).replace("", None).cast(pl.Float64) for c in numeric_columns]
)

lf = lf.rename(
    {
        "ADU": "ADUANA",
        "FECHA_": "FECHA",
        "NOMBRE_IMPORTADOR": "IMPORTADOR",
        "M": "MEDIO_TRANSPORTE",
        "UN": "UNIDAD_MEDIDA",
        "CANT_UNIDAD_MEDIDA": "CANTIDAD_UNIDAD_MEDIDA",
        "FOB_DOLAR": "FOB_UNITARIO_USD",
        "FOB_TOTAL": "FOB_TOTAL_USD",
        "DIV": "DIVISA",
        "POS_NCM": "NCM",
    }
)

lf.sink_parquet(output_path)
logger.info(f"Wrote decoded dataset to {output_path}")

# Preview from the written output rather than re-running the decode pipeline: reading
# back from disk only re-scans the already-decoded, compressed result.
decoded = pl.scan_parquet(output_path)
print(decoded.head().collect())
print(
    decoded.select(
        [
            "ADUANA",
            "MEDIO_TRANSPORTE",
            "UNIDAD_MEDIDA",
            "DIVISA",
            "PAIS_ORIGEN",
            "PAIS_PROCEDENCIA",
        ]
    )
    .null_count()
    .collect()
)

Wrote decoded dataset to ..\data\processed\importaciones_2019-2026_decoded.parquet


shape: (5, 15)
┌────────────┬────────────┬──────────┬────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ ADUANA     ┆ DESTINACIO ┆ NUM_ITEM ┆ FECHA  ┆ … ┆ PAIS_ORIGE ┆ PAIS_PROC ┆ NCM       ┆ MONTO_TRI │
│ ---        ┆ N          ┆ ---      ┆ ---    ┆   ┆ N          ┆ EDENCIA   ┆ ---       ┆ BUTADO_TO │
│ str        ┆ ---        ┆ str      ┆ str    ┆   ┆ ---        ┆ ---       ┆ str       ┆ TAL       │
│            ┆ str        ┆          ┆        ┆   ┆ str        ┆ str       ┆           ┆ ---       │
│            ┆            ┆          ┆        ┆   ┆            ┆           ┆           ┆ f64       │
╞════════════╪════════════╪══════════╪════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ BS.AS.(CAP ┆ 19001IC040 ┆ 5        ┆ 201901 ┆ … ┆ TAILANDIA  ┆ SINGAPUR  ┆ 9401.90.9 ┆ 104.19    │
│ ITAL)      ┆ 01532X     ┆          ┆        ┆   ┆            ┆           ┆ 0         ┆           │
│ BS.AS.(CAP ┆ 19001IC040 ┆ 4        ┆ 201901 ┆ … ┆ TAILANDIA  ┆ SINGAPUR  ┆